# **Predicting Students' Dropout and Academic Success**
**Team / Authors:** RxPulseMH — Deocampo, Jan Adrian | Jalbuena, Jerica Thea | Julabar, Jay Adrian | Maravilla, Louiena Paula | Brendia, Ricky Boy

**Course:** ICT 110  
**Date:** November 10, 2026  

**Result Summary:** This study frames student retention at the Polytechnic Institute of Portalegre as a supervised binary classification task (`Dropout` vs. `Graduate`) across 3,630 student records and 36 demographic, socioeconomic, and two-semester academic features. A naive majority-class baseline (`Graduate`) achieves 60.85% accuracy but only a 0.3783 Macro-F1 score (0.0000 F1 on `Dropout`), establishing the minimum performance floor that subsequent machine learning models must outperform.

---
# **A. Problem Definition**

### **Brief Description**
This project utilizes the **Predict Students' Dropout and Academic Success** dataset from the UCI Machine Learning Repository (Dataset ID: 697), collected from the Polytechnic Institute of Portalegre in Portugal across academic years 2008/2009 to 2018/2019. The dataset integrates demographic, socioeconomic, macroeconomic, admission, and academic performance records from the first and second semesters across 36 predictor attributes under the Creative Commons Attribution 4.0 International (CC BY 4.0) license.

### **Problem Statement**
Student dropout negatively impacts institutional success and graduation rates. Identifying at-risk students allows academic administrators and advisers to provide timely, targeted support. This project utilizes supervised binary classification to predict whether a student will ultimately graduate or drop out. By leveraging a comprehensive timeline of demographic, socioeconomic, and academic performance features across two semesters, the model aims to accurately classify eventual academic outcomes and assist in the university's retention efforts.

### **Guiding Questions (Stage A)**
* **A2 — Problem Framing:** This study is framed as **supervised binary classification**. Because historical student records are labeled with known academic outcomes and the operational goal is to categorize students into one of two discrete, mutually exclusive states (`Dropout` or `Graduate`), binary classification directly fits the decision-support objective.
* **A3 — Target Variable and Measurement:** The outcome of interest is the `Target` variable, which records each student's definitive academic status (`Graduate`: 2,209 students, `Dropout`: 1,421 students) measured by the institution at the end of the normal duration of their course.
* **A4 — Evaluation Metric and Cost of Errors:** Success is defined by the **Macro-Averaged F1-Score (Macro-F1)**. Because the binary target distribution is imbalanced (60.85% `Graduate` vs. 39.15% `Dropout`), standard accuracy naturally favors the majority class. In an academic advising context, a **False Negative** (failing to identify an actual `Dropout`) deprives an at-risk student of timely support, while a **False Positive** (incorrectly flagging a `Graduate` as at-risk) misallocates limited institutional advising resources. Macro-F1 computes the F1-score for both classes independently and averages them equally, ensuring balanced precision and recall across both outcomes.
* **A7 — Naive Baseline Score:** A naive majority-class baseline (`DummyClassifier`) that predicts `Graduate` for all students achieves **60.85% accuracy** and a **Macro-F1 score of 0.3783** (averaging `0.7566` F1 for `Graduate` and `0.0000` F1 for `Dropout`). Alongside this naive floor, an untuned **Logistic Regression** model will serve as a basic mathematical baseline in the modeling stage.

### **1. Packages and Random Seeds**
We install the UCI repository helper library, import the required data manipulation and evaluation packages, and set a fixed global random seed (`SEED = 42`) to ensure full reproducibility across all runs.

In [ ]:
!pip install -q ucimlrepo

import random
import sys
import numpy as np
import pandas as pd
import sklearn
from ucimlrepo import fetch_ucirepo
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Set and print random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Global Random Seed set to: {SEED}")

Global Random Seed set to: 42


**Output Interpretation:** All required libraries are loaded and the global random seed is locked at `42`, guaranteeing deterministic splits and cross-validation results from top to bottom.

### **2. Dataset Loading and Scope Inspection (A3)**
We fetch the dataset (ID: 697) directly from the UCI Machine Learning Repository, scope the dataset to our binary target outcomes (`Graduate` and `Dropout`), and inspect the resulting shape, feature count, and class distribution.

In [ ]:
# Load dataset from UCI ML Repository (ID: 697)
uci_dataset = fetch_ucirepo(id=697)
df_all = pd.concat([uci_dataset.data.features, uci_dataset.data.targets], axis=1)

# Scope dataset to the binary classification outcomes (Graduate vs. Dropout)
df = df_all[df_all['Target'].isin(['Graduate', 'Dropout'])].copy().reset_index(drop=True)

X = df.drop(columns=['Target'])
y = df['Target']

class_dist = pd.DataFrame({
    'Count': y.value_counts(),
    'Percentage (%)': (y.value_counts(normalize=True) * 100).round(2)
})

print(f"Dataset Shape (Records, Columns): {df.shape}")
print(f"Predictor Features: {X.shape[1]} | Target Variable: 1 ('Target')\n")
print("Target Class Distribution (A3):")
print(class_dist)

Dataset Shape (Records, Columns): (3630, 37)
Predictor Features: 36 | Target Variable: 1 ('Target')

Target Class Distribution (A3):
          Count  Percentage (%)
Target                         
Graduate   2209           60.85
Dropout    1421           39.15


**Output Interpretation:** The dataset contains 3,630 student records across 36 predictor features and 1 binary target variable (`Target`), exhibiting a moderate class imbalance with 2,209 `Graduate` students (60.85%) and 1,421 `Dropout` students (39.15%).

### **3. Preview of Student Records**
We display the first five rows of the loaded dataframe to verify that demographic, socioeconomic, and two-semester academic features alongside the `Target` label are properly structured.

In [ ]:
# Inspect a subset of key columns across the first 5 records
sample_cols = [
    'Marital Status', 'Application mode', 'Application order', 'Course',
    'Daytime/evening attendance', 'Previous qualification',
    'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Target'
]
display(df[sample_cols].head())


,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Target
0,1,17,5,171,1,1,0,0.000000,Dropout
1,1,15,1,9254,1,1,6,13.666667,Graduate
2,1,1,5,9070,1,1,0,0.000000,Dropout
3,1,17,2,9773,1,1,5,12.400000,Graduate
4,2,39,1,8014,0,1,6,13.000000,Graduate


**Output Interpretation:** The sample records confirm that all predictor columns are numerically encoded and aligned with the categorical `Target` outcome (`Dropout` vs. `Graduate`).

### **4. Naive Baseline Evaluation (A7)**
We evaluate a Majority Class (`DummyClassifier`) baseline using 5-fold stratified cross-validation to establish the minimum Macro-F1 performance floor and demonstrate why standard accuracy is insufficient for imbalanced retention data.

In [ ]:
# Initialize Majority Class (Dummy) Classifier
dummy_clf = DummyClassifier(strategy='most_frequent', random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Compute 5-fold cross-validated Accuracy and Macro-F1
cv_acc = cross_val_score(dummy_clf, X, y, cv=cv, scoring='accuracy').mean()
cv_macro_f1 = cross_val_score(dummy_clf, X, y, cv=cv, scoring='f1_macro').mean()

# Fit on dataset to display the per-class breakdown
dummy_clf.fit(X, y)
y_pred_dummy = dummy_clf.predict(X)

print("=== Naive Baseline (Majority Class = 'Graduate') ===")
print(f"5-Fold CV Accuracy : {cv_acc:.4f} ({cv_acc*100:.2f}%)")
print(f"5-Fold CV Macro-F1 : {cv_macro_f1:.4f}\n")
print("Per-Class Performance Breakdown:")
print(classification_report(y, y_pred_dummy, digits=4, zero_division=0))

=== Naive Baseline (Majority Class = 'Graduate') ===
5-Fold CV Accuracy : 0.6085 (60.85%)
5-Fold CV Macro-F1 : 0.3783

Per-Class Performance Breakdown:
              precision    recall  f1-score   support

     Dropout     0.0000    0.0000    0.0000      1421
    Graduate     0.6085    1.0000    0.7566      2209

    accuracy                         0.6085      3630
   macro avg     0.3043    0.5000    0.3783      3630
weighted avg     0.3703    0.6085    0.4604      3630



**Output Interpretation:** Predicting `Graduate` for every student yields a deceptive 60.85% accuracy while completely failing to detect any dropouts (`0.0000` F1 on `Dropout`), resulting in a naive baseline **Macro-F1 score of 0.3783**.